# SpendDNA – Your Wallet's Story

### Industry-Graded Minor Project – Data Science

**Name:** SHABNAM
**Batch:** JULY BATCH


> Spotify Wrapped for your money.

SpendDNA analyzes six months of synthetic Indian transaction data using
Python, NumPy and Pandas.

## Mandatory Features

1. Transaction Parser
2. Vendor Extractor
3. Category Tagger
4. Spending Overview
5. Monthly Spending Trend
6. Time-of-Day Patterns
7. Anomaly Detection
8. Spending Archetype Detection

## Bonus Features

- Day-of-week analysis
- Vendor cleanup audit
- Custom archetype
- NumPy-based spending forecast

### AI Assistance Disclosure

AI assistance was used for syntax support, debugging and code review.
Dataset-specific vendor mappings and analytical results were verified
using the provided dataset.

In [1]:
# ============================================================
# SpendDNA
# SETUP
# ============================================================

import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

FILE_NAME = next(iter(uploaded))

df = pd.read_csv(FILE_NAME)

print("File loaded:", FILE_NAME)
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

Saving Data set for DADS June.csv to Data set for DADS June.csv
File loaded: Data set for DADS June.csv
Rows: 1328
Columns: 8


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


# Feature 1 – Transaction Parser

The transaction parser cleans the raw bank/UPI export by:

- Handling the four date formats
- Handling the three amount formats
- Standardising DR/CR and Debit/Credit
- Removing exact duplicate transactions
- Handling missing values
- Extracting hour, month and day-of-week information

In [3]:
# ============================================================
# FEATURE 1 - TRANSACTION PARSER
# ============================================================

print("=" * 70)
print("FEATURE 1 - TRANSACTION PARSER")
print("=" * 70)


# ------------------------------------------------------------
# REMOVE EXACT DUPLICATES
# ------------------------------------------------------------

rows_before = len(df)

df = df.drop_duplicates().copy()

duplicates_removed = (
    rows_before - len(df)
)


# ------------------------------------------------------------
# PARSE DATE
# ------------------------------------------------------------

df["date"] = pd.to_datetime(
    df["Date"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)


# ------------------------------------------------------------
# PARSE AMOUNT
# ------------------------------------------------------------

df["amount"] = (
    df["Amount"]
    .astype(str)
    .str.replace(
        "₹",
        "",
        regex=False
    )
    .str.replace(
        "Rs.",
        "",
        regex=False
    )
    .str.replace(
        ",",
        "",
        regex=False
    )
    .str.strip()
)

df["amount"] = pd.to_numeric(
    df["amount"],
    errors="coerce"
)


# ------------------------------------------------------------
# STANDARDISE TYPE
# ------------------------------------------------------------

df["type"] = (
    df["Type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["type"] = df["type"].replace({

    "dr": "debit",

    "cr": "credit",

    "debit": "debit",

    "credit": "credit"
})


# ------------------------------------------------------------
# HANDLE EMPTY MODE VALUES
# ------------------------------------------------------------

df["Mode"] = df["Mode"].replace(
    "",
    np.nan
)


# ------------------------------------------------------------
# CHECK INVALID VALUES
# ------------------------------------------------------------

invalid_dates = df["date"].isna().sum()

invalid_amounts = df["amount"].isna().sum()


# Remove invalid date/amount rows

df = df.dropna(
    subset=[
        "date",
        "amount"
    ]
).copy()


# ------------------------------------------------------------
# CREATE DATETIME FEATURES
# ------------------------------------------------------------

df["hour"] = (
    df["Time"]
    .astype(str)
    .str[:2]
    .astype(int)
)

df["month"] = (
    df["date"]
    .dt.strftime("%b")
)

df["month_num"] = (
    df["date"]
    .dt.month
)

df["day_of_week"] = (
    df["date"]
    .dt.day_name()
)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print(
    f"Raw transactions       : {rows_before}"
)

print(
    f"Duplicate rows removed : {duplicates_removed}"
)

print(
    f"Clean transactions     : {len(df)}"
)

print(
    f"Invalid dates          : {invalid_dates}"
)

print(
    f"Invalid amounts        : {invalid_amounts}"
)

print(
    f"\nDate dtype   : {df['date'].dtype}"
)

print(
    f"Amount dtype : {df['amount'].dtype}"
)

print("\nCleaned data:")
display(df.head())

FEATURE 1 - TRANSACTION PARSER
Raw transactions       : 1328
Duplicate rows removed : 18
Clean transactions     : 1310
Invalid dates          : 0
Invalid amounts        : 0

Date dtype   : datetime64[ns]
Amount dtype : float64

Cleaned data:


,Date,Time,Description,Type,Amount,Balance,Mode,Ref,date,amount,type,hour,month,month_num,day_of_week
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872,2024-01-01,2462.0,debit,3,Jan,1,Monday
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064,2024-01-01,50.0,debit,5,Jan,1,Monday
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316,2024-01-01,84728.0,credit,9,Jan,1,Monday
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226,2024-01-01,1828.0,debit,14,Jan,1,Monday
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962,2024-01-01,270.0,debit,14,Jan,1,Monday


# Feature 2 – Vendor Extractor

The vendor extractor converts messy transaction descriptions into canonical vendor names.

Examples:

- `POS SWIGGY BANGALORE` → Swiggy
- `BUNDL Tech P L` → Swiggy
- `BHIM ZEPTO` → Zepto
- `AMZN-INTPYMT` → Amazon
- `UPI-ZERODHA-COIN@AXIS` → Zerodha
- `ATM-WDL-HDFC-3609` → Cash Withdrawal
- `UPI-AMAN-8934@OKAXIS` → P2P Transfer

No regular expressions are used.

In [4]:
# ============================================================
# FEATURE 2 - INSPECT DESCRIPTIONS
# ============================================================

print("=" * 70)
print("FEATURE 2 - VENDOR EXTRACTION")
print("=" * 70)

print(
    "Number of unique descriptions:",
    df["Description"].nunique()
)

print("\nSample descriptions:")

for description in df["Description"].unique()[:40]:

    print("-", description)

FEATURE 2 - VENDOR EXTRACTION
Number of unique descriptions: 283

Sample descriptions:
- AMAZON SELLER SVCS
- BHIM-BMTC
- NEFT-TECHCRUSH LABS-SALARY MAY24
- UPI-AMAN-8934@OKAXIS
- BHIM-BLINKIT
- BHIM ZEPTO
- UPI-UBER-2426@HDFCBANK
- POS SWIGGY BANGALORE
- UPI-GROWWPAY@HDFCBANK
- OLA ELECTRIC
- BMS MOVIE TICKETS
- POS OLA-PRIME
- SWIGGY-INSTAMART
- UPI-STARBUCKS@AXIS
- UPI-THIRDWAVE@OKAXIS
- ANI Technologies
- BMTC BUS PASS
- POS TRUFFLES
- FLIPKART INDIA
- POS SWIGGY-RESTAURANT
- GROFERS INDIA P L
- POS UBER BANGALORE
- BANGALORE ELEC SUPPLY
- TWC INDIA
- UPI-BESCOM-BILL@HDFCBANK
- UPI-AMAN-0816@OKAXIS
- ROPPEN TRANSPORTATION
- OLA CABS
- POS ZOMATO
- UPI-AMAZONPAY@HDFCBANK
- POS BLINKIT
- IMPS-RENT-LANDLORD-75500265
- ZOMATO MEDIA P L
- UPI-ANKIT-6430@OKAXIS
- UPI-OLACABS@HDFCBANK
- UPI-JIORECHARGE@PAYTM
- UPI-CCD@HDFCBANK
- Swiggy*Order
- INSTAMART BANGALORE
- UPI-ZOMATO-LIMITED@PAYTM


In [5]:
# ============================================================
# VENDOR DICTIONARY
# ============================================================

vendor_keywords = {

    # --------------------------------------------------------
    # SPECIAL TRANSACTION TYPES
    # --------------------------------------------------------

    "Cash Withdrawal": [
        "ATM-WDL"
    ],

    "P2P Transfer": [
        "UPI-AMAN-",
        "UPI-ANKIT-",
        "UPI-PRIYA-",
        "UPI-NEHA-",
        "UPI-VIKAS-",
        "UPI-KARAN-",
        "UPI-SNEHA-"
    ],

    "Salary": [
        "TECHCRUSH LABS-SALARY"
    ],

    "Rent": [
        "IMPS-RENT-LANDLORD"
    ],


    # --------------------------------------------------------
    # QUICK COMMERCE SPECIAL CASES
    # --------------------------------------------------------

    "Instamart": [
        "SWIGGY-INSTAMART",
        "BUNDL TECH-INSTAMART",
        "POS INSTAMART",
        "INSTAMART BANGALORE"
    ],


    # --------------------------------------------------------
    # AMAZON PRIME BEFORE AMAZON
    # --------------------------------------------------------

    "Amazon Prime": [
        "AMAZON PRIME",
        "AMZN PRIME",
        "AMAZON-PRIME"
    ],


    # --------------------------------------------------------
    # FOOD DELIVERY
    # --------------------------------------------------------

    "Swiggy": [
        "SWIGGY",
        "BUNDL TECH P L"
    ],

    "Zomato Dining": [
        "ZOMATO-DINING"
    ],

    "Zomato": [
        "ZOMATO"
    ],


    # --------------------------------------------------------
    # QUICK COMMERCE
    # --------------------------------------------------------

    "Zepto": [
        "ZEPTO"
    ],

    "Blinkit": [
        "BLINKIT"
    ],


    # --------------------------------------------------------
    # E-COMMERCE
    # --------------------------------------------------------

    "Amazon": [
        "AMAZON",
        "AMZN"
    ],

    "Flipkart": [
        "FLIPKART",
        "FKART"
    ],

    "Myntra": [
        "MYNTRA"
    ],

    "Nykaa": [
        "NYKAA",
        "FSN E-COMMERCE"
    ],


    # --------------------------------------------------------
    # GROCERIES
    # --------------------------------------------------------

    "D-Mart": [
        "DMART",
        "AVENUE SUPERMARTS"
    ],

    "BigBasket": [
        "BIGBASKET"
    ],

    "Grofers": [
        "GROFERS"
    ],

    "KiranaKart": [
        "KIRANAKART"
    ],

    "Innovative Retail": [
        "INNOVATIVE RETAIL"
    ],


    # --------------------------------------------------------
    # INVESTMENTS
    # --------------------------------------------------------

    "Zerodha": [
        "ZERODHA"
    ],

    "Groww": [
        "GROWW"
    ],


    # --------------------------------------------------------
    # TRANSPORT
    # --------------------------------------------------------

    "Uber": [
        "UBER"
    ],

    "Ola": [
        "OLA CABS",
        "OLA-PRIME",
        "OLA ELECTRIC",
        "ANI TECHNOLOGIES"
    ],

    "Rapido": [
        "RAPIDO"
    ],

    "BMTC": [
        "BMTC",
        "TUMMOC"
    ],

    "Roppen": [
        "ROPPEN TRANSPORTATION"
    ],


    # --------------------------------------------------------
    # CAFE
    # --------------------------------------------------------

    "Starbucks": [
        "STARBUCKS",
        "TATA STARBUCKS"
    ],

    "Third Wave Coffee": [
        "THIRDWAVE",
        "THIRD WAVE",
        "TWC INDIA"
    ],

    "Cafe Coffee Day": [
        "COFFEE DAY",
        "CCD"
    ],


    # --------------------------------------------------------
    # RESTAURANTS
    # --------------------------------------------------------

    "Truffles": [
        "TRUFFLES"
    ],

    "Meghana Foods": [
        "MEGHANA FOODS"
    ],

    "Empire Restaurant": [
        "EMPIRE RESTAURANT"
    ],

    "Bangalore Restaurant": [
        "BANGALORE RESTAURANT"
    ],

    "Dineout": [
        "DINEOUT"
    ],

    "Restaurant": [
        "UPI-RESTAURANT"
    ],


    # --------------------------------------------------------
    # FUEL
    # --------------------------------------------------------

    "HP Petrol": [
        "HP PETROL"
    ],

    "BPCL": [
        "BPCL"
    ],

    "Indian Oil": [
        "INDIAN OIL",
        "UPI-IOC",
        "IOC"
    ],


    # --------------------------------------------------------
    # ENTERTAINMENT
    # --------------------------------------------------------

    "Netflix": [
        "NETFLIX"
    ],

    "Spotify": [
        "SPOTIFY"
    ],

    "Disney+ Hotstar": [
        "DISNEY HOTSTAR",
        "HOTSTAR"
    ],

    "BookMyShow": [
        "BOOKMYSHOW",
        "BMS MOVIE TICKETS"
    ],

    "Star India": [
        "STAR INDIA"
    ],

    "BigTree Entertainment": [
        "BIGTREE ENTERTAINMENT"
    ],


    # --------------------------------------------------------
    # UTILITIES
    # --------------------------------------------------------

    "Airtel": [
        "AIRTEL",
        "BHARTI AIRTEL"
    ],

    "Vi": [
        "VI POSTPAID",
        "VODAFONE IDEA",
        "UPI-VI-RECHARGE"
    ],

    "Jio": [
        "JIOFIBER",
        "RELIANCE JIO",
        "JIORECHARGE"
    ],

    "BESCOM": [
        "BESCOM",
        "BANGALORE ELEC SUPPLY"
    ],

    "BWSSB": [
        "BWSSB"
    ]
}


# ============================================================
# VENDOR EXTRACTION FUNCTION
# ============================================================

def extract_vendor(description):

    text = str(description).upper()

    for vendor in vendor_keywords:

        keywords = vendor_keywords[vendor]

        for keyword in keywords:

            if keyword in text:

                return vendor

    return "Uncategorised"


# ============================================================
# APPLY FUNCTION
# ============================================================

df["vendor_clean"] = (
    df["Description"]
    .apply(extract_vendor)
)


print("Vendor extraction completed.")

Vendor extraction completed.


In [6]:
# ============================================================
# VENDOR RESULTS
# ============================================================

print("=" * 70)
print("VENDOR EXTRACTION RESULTS")
print("=" * 70)

print(
    "Canonical vendors:",
    df["vendor_clean"].nunique()
)

print("\nTop 10 vendors:")

print(
    df["vendor_clean"]
    .value_counts()
    .head(10)
)


# ------------------------------------------------------------
# UNCATEGORISED AUDIT
# ------------------------------------------------------------

uncategorised = (
    df[
        df["vendor_clean"] ==
        "Uncategorised"
    ]["Description"]
    .unique()
)


print(
    "\nUncategorised descriptions:",
    len(uncategorised)
)


for item in uncategorised:

    print(
        "-",
        item
    )

VENDOR EXTRACTION RESULTS
Canonical vendors: 51

Top 10 vendors:
vendor_clean
Swiggy       176
Zomato       101
Ola           77
Amazon        76
Uber          71
Instamart     67
Zepto         58
Flipkart      47
Starbucks     42
Rapido        41
Name: count, dtype: int64

Uncategorised descriptions: 1
- UPI-OLACABS@HDFCBANK


# Feature 3 – Category Tagger

Every transaction is assigned to a spending category.

The required categories are:

- Food Delivery
- Quick Commerce
- E-commerce
- Transport
- Cafe
- Restaurants
- Subscriptions
- Utilities
- Groceries
- Investments
- Fuel
- Entertainment

P2P transfers and cash withdrawals are maintained separately.

In [7]:
# ============================================================
# FEATURE 3 - CATEGORY TAGGER
# ============================================================


category_map = {

    # Food Delivery
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",


    # Quick Commerce
    "Instamart": "Quick Commerce",
    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",


    # E-commerce
    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",


    # Transport
    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",
    "Roppen": "Transport",


    # Cafe
    "Starbucks": "Cafe",
    "Third Wave Coffee": "Cafe",
    "Cafe Coffee Day": "Cafe",


    # Restaurants
    "Truffles": "Restaurants",
    "Meghana Foods": "Restaurants",
    "Empire Restaurant": "Restaurants",
    "Bangalore Restaurant": "Restaurants",
    "Dineout": "Restaurants",
    "Zomato Dining": "Restaurants",
    "Restaurant": "Restaurants",


    # Subscriptions
    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "Disney+ Hotstar": "Subscriptions",
    "Amazon Prime": "Subscriptions",


    # Utilities
    "Airtel": "Utilities",
    "Vi": "Utilities",
    "Jio": "Utilities",
    "BESCOM": "Utilities",
    "BWSSB": "Utilities",
    "Rent": "Utilities",


    # Groceries
    "D-Mart": "Groceries",
    "BigBasket": "Groceries",
    "Grofers": "Groceries",
    "KiranaKart": "Groceries",
    "Innovative Retail": "Groceries",


    # Investments
    "Zerodha": "Investments",
    "Groww": "Investments",


    # Fuel
    "HP Petrol": "Fuel",
    "BPCL": "Fuel",
    "Indian Oil": "Fuel",


    # Entertainment
    "BookMyShow": "Entertainment",
    "Star India": "Entertainment",
    "BigTree Entertainment": "Entertainment",


    # Special categories
    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal",


    # Income
    "Salary": "Income"
}


df["category"] = (
    df["vendor_clean"]
    .map(category_map)
    .fillna("Uncategorised")
)


print("=" * 70)
print("FEATURE 3 - CATEGORY TAGGER")
print("=" * 70)

print(
    df["category"]
    .value_counts()
)

FEATURE 3 - CATEGORY TAGGER
category
Food Delivery        277
Transport            240
Quick Commerce       165
E-commerce           162
Cafe                  99
Restaurants           93
Groceries             69
Utilities             49
Subscriptions         38
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         16
Uncategorised         10
Income                 6
Name: count, dtype: int64


In [8]:
# ============================================================
# CATEGORY VERIFICATION
# ============================================================


required_categories = [

    "Food Delivery",
    "Quick Commerce",
    "E-commerce",
    "Transport",
    "Cafe",
    "Restaurants",
    "Subscriptions",
    "Utilities",
    "Groceries",
    "Investments",
    "Fuel",
    "Entertainment"
]


missing_categories = []


for category in required_categories:

    if category not in df["category"].unique():

        missing_categories.append(category)


print(
    "Required categories:",
    len(required_categories)
)

print(
    "Missing categories:",
    missing_categories
)


if len(missing_categories) == 0:

    print(
        "\nAll 12 required categories are present."
    )

else:

    print(
        "\nSome required categories are missing."
    )

Required categories: 12
Missing categories: []

All 12 required categories are present.


# Feature 4 – Spending Overview

This section calculates:

- Total credits
- Total debits
- Net change
- Savings rate
- Total transactions
- Unique vendors
- Top 5 spending categories
- Top 5 vendors

Personal transfers and cash withdrawals are excluded from consumption percentage calculations.

In [9]:
# ============================================================
# FEATURE 4 - SPENDING OVERVIEW
# ============================================================


credits = df[
    df["type"] == "credit"
].copy()


debits = df[
    df["type"] == "debit"
].copy()


total_credits = credits["amount"].sum()

total_debits = debits["amount"].sum()


net_change = (
    total_credits -
    total_debits
)


if total_credits != 0:

    savings_rate = (
        net_change /
        total_credits
        *
        100
    )

else:

    savings_rate = 0


# ------------------------------------------------------------
# CONSUMPTION DATA
# ------------------------------------------------------------

consumption = debits[
    ~debits["category"].isin([

        "Personal Transfer",

        "Cash Withdrawal",

        "Uncategorised"
    ])
].copy()


# ------------------------------------------------------------
# CATEGORY SPEND
# ------------------------------------------------------------

category_spend = (

    consumption
    .groupby("category")["amount"]
    .sum()
    .sort_values(
        ascending=False
    )
)


# ------------------------------------------------------------
# VENDOR SPEND
# ------------------------------------------------------------

vendor_spend = (

    consumption
    .groupby("vendor_clean")["amount"]
    .sum()
    .sort_values(
        ascending=False
    )
)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print("=" * 70)
print("FEATURE 4 - SPENDING OVERVIEW")
print("=" * 70)

print(
    f"Total credits : ₹{total_credits:,.2f}"
)

print(
    f"Total debits  : ₹{total_debits:,.2f}"
)

print(
    f"Net change    : ₹{net_change:,.2f}"
)

print(
    f"Savings rate  : {savings_rate:.2f}%"
)

print(
    f"Transactions  : {len(df)}"
)

print(
    f"Unique vendors: "
    f"{df['vendor_clean'].nunique()}"
)


print("\nTOP 5 CATEGORIES")


for category, amount in (
    category_spend
    .head(5)
    .items()
):

    percentage = (
        amount /
        total_debits
        *
        100
    )

    print(

        f"{category:<20}"
        f"{percentage:>6.2f}% "
        f"₹{amount:>12,.2f}"
    )


print("\nTOP 5 VENDORS")


for vendor, amount in (
    vendor_spend
    .head(5)
    .items()
):

    transaction_count = (
        consumption[
            consumption["vendor_clean"] ==
            vendor
        ].shape[0]
    )

    print(

        f"{vendor:<22}"
        f"₹{amount:>12,.2f} "
        f"({transaction_count} transactions)"
    )

FEATURE 4 - SPENDING OVERVIEW
Total credits : ₹509,774.00
Total debits  : ₹1,678,901.00
Net change    : ₹-1,169,127.00
Savings rate  : -229.34%
Transactions  : 1310
Unique vendors: 51

TOP 5 CATEGORIES
E-commerce           35.37% ₹  593,769.00
Investments          14.78% ₹  248,160.00
Utilities             8.93% ₹  149,914.00
Restaurants           7.58% ₹  127,290.00
Food Delivery         7.12% ₹  119,501.00

TOP 5 VENDORS
Amazon                ₹  318,422.00 (76 transactions)
Zerodha               ₹  210,000.00 (14 transactions)
Flipkart              ₹  177,510.00 (47 transactions)
Rent                  ₹  108,000.00 (6 transactions)
Swiggy                ₹   73,738.00 (176 transactions)


# Feature 5 – Monthly Spending Trend

This feature creates a category × month spending matrix and calculates month-on-month changes.

The analysis identifies:

- Monthly category spending
- Biggest month-on-month growth
- Biggest month-on-month decline

In [10]:
# ============================================================
# FEATURE 5 - MONTHLY TREND ANALYSIS
# ============================================================


month_order = [

    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun"
]


monthly_matrix = (

    consumption

    .pivot_table(

        values="amount",

        index="category",

        columns="month_num",

        aggfunc="sum",

        fill_value=0
    )

    .reindex(

        columns=range(1, 7),

        fill_value=0
    )
)


monthly_matrix.columns = month_order


print("=" * 70)
print("FEATURE 5 - MONTHLY SPENDING TREND")
print("=" * 70)

display(
    monthly_matrix.round(2)
)


# ------------------------------------------------------------
# MONTH-ON-MONTH GROWTH
# ------------------------------------------------------------

trend_records = []


for category in monthly_matrix.index:

    for i in range(1, 6):

        previous = (
            monthly_matrix
            .loc[
                category,
                month_order[i - 1]
            ]
        )

        current = (
            monthly_matrix
            .loc[
                category,
                month_order[i]
            ]
        )


        if previous != 0:

            change = (

                (
                    current -
                    previous
                )
                /
                previous
                *
                100
            )


            trend_records.append({

                "category":
                    category,

                "from_month":
                    month_order[i - 1],

                "to_month":
                    month_order[i],

                "change_pct":
                    change
            })


trend_df = pd.DataFrame(
    trend_records
)


biggest_growth = (
    trend_df
    .sort_values(
        "change_pct",
        ascending=False
    )
    .iloc[0]
)


biggest_decline = (
    trend_df
    .sort_values(
        "change_pct",
        ascending=True
    )
    .iloc[0]
)


print("\nBIGGEST MONTH-ON-MONTH GROWTH")

print(

    f"{biggest_growth['category']} | "
    f"{biggest_growth['from_month']} → "
    f"{biggest_growth['to_month']} | "
    f"{biggest_growth['change_pct']:.2f}%"
)


print("\nBIGGEST MONTH-ON-MONTH DECLINE")

print(

    f"{biggest_decline['category']} | "
    f"{biggest_decline['from_month']} → "
    f"{biggest_decline['to_month']} | "
    f"{biggest_decline['change_pct']:.2f}%"
)

FEATURE 5 - MONTHLY SPENDING TREND


,Jan,Feb,Mar,Apr,May,Jun
category,,,,,,
Cafe,3690.0,4273.0,5448.0,6564.0,5668.0,5802.0
E-commerce,97134.0,92773.0,103772.0,68098.0,95001.0,136991.0
Entertainment,1263.0,1746.0,2856.0,3366.0,0.0,1914.0
Food Delivery,20206.0,18714.0,19166.0,20726.0,21193.0,19496.0
Fuel,30322.0,2079.0,26164.0,18718.0,9138.0,2882.0
Groceries,20593.0,9849.0,6971.0,13773.0,13546.0,8468.0
Investments,38432.0,15000.0,68644.0,54126.0,48628.0,23330.0
Quick Commerce,9853.0,16187.0,17297.0,14547.0,11360.0,12630.0
Restaurants,17004.0,24510.0,29997.0,10039.0,23260.0,22480.0



BIGGEST MONTH-ON-MONTH GROWTH
Fuel | Feb → Mar | 1158.49%

BIGGEST MONTH-ON-MONTH DECLINE
Entertainment | Apr → May | -100.00%


# Feature 6 – Time-of-Day Spending Patterns

This feature analyses spending by hour of the day.

A text-based heatmap is used because the project explicitly prohibits matplotlib and seaborn.

The analysis also identifies:

- Peak hour for each category
- Late-night Food Delivery percentage
- Category × hour spending patterns

In [11]:
# ============================================================
# FEATURE 6 - TIME-OF-DAY PATTERNS
# ============================================================


time_matrix = (

    consumption

    .pivot_table(

        values="amount",

        index="category",

        columns="hour",

        aggfunc="sum",

        fill_value=0
    )

    .reindex(

        columns=range(24),

        fill_value=0
    )
)


print("=" * 70)
print("FEATURE 6 - TIME-OF-DAY PATTERNS")
print("=" * 70)


# ------------------------------------------------------------
# ASCII HEATMAP
# ------------------------------------------------------------

print(
    "Hour:",
    " ".join(
        f"{hour:02d}"
        for hour in range(24)
    )
)


for category in time_matrix.index:

    values = (
        time_matrix
        .loc[category]
    )

    maximum = values.max()

    bars = []


    for value in values:

        if maximum == 0:

            bars.append(".")


        else:

            length = int(
                value /
                maximum *
                6
            )

            if (
                length == 0
                and
                value > 0
            ):

                length = 1


            if length > 0:

                bars.append(
                    "#" * length
                )

            else:

                bars.append(".")


    print(

        f"{category:<20}",

        " ".join(
            f"{bar:>6}"
            for bar in bars
        )
    )


# ------------------------------------------------------------
# PEAK HOUR
# ------------------------------------------------------------

print("\nPEAK HOUR BY CATEGORY")


for category in time_matrix.index:

    peak_hour = (

        time_matrix
        .loc[category]
        .idxmax()
    )


    print(

        f"{category:<20}"
        f"{int(peak_hour):02d}:00"
    )


# ------------------------------------------------------------
# LATE NIGHT FOOD DELIVERY
# ------------------------------------------------------------

food_delivery = consumption[
    consumption["category"] ==
    "Food Delivery"
].copy()


late_food = food_delivery[
    (
        food_delivery["hour"] >= 21
    )
    |
    (
        food_delivery["hour"] <= 2
    )
]


if len(food_delivery) > 0:

    late_food_percentage = (

        len(late_food)
        /
        len(food_delivery)
        *
        100
    )

else:

    late_food_percentage = 0


print(

    f"\nLate-night Food Delivery "
    f"(21:00–02:59): "
    f"{late_food_percentage:.2f}%"
)

FEATURE 6 - TIME-OF-DAY PATTERNS
Hour: 00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Cafe                      #      #      .      #      .      .      #      #    ###     ## ######     ##      #     ##      #     ##   ####  #####    ###      #     ##      .      .      #
E-commerce                #      #      #      #      #      #      #      #      #    ###      #   ####      #      # ######      #   ####     ##     ##      #   ####     ##      #      #
Entertainment             #      #      .  #####    ###      .      .      #   ####      .      .      .      .      .      .      . ######      .   ####      #   ####      .      #      .
Food Delivery             #      #      #      #      #      #      #      #      #      #      #    ###    ###      #      #     ##      #      #  #####  ##### ######    ###     ##      #
Fuel                      #      #      .      #      #      #      .      .      #      #      .      #  #####   ####    ### ###### 

# Feature 7 – Anomaly Detection

Anomalies are detected using a manually calculated z-score.

For each spending category:

z = (amount − category mean) / category standard deviation

Transactions with z-score greater than 2 are flagged as anomalous.

No scipy or machine-learning library is used.

In [ ]:
# ============================================================
# FEATURE 7 - ANOMALY DETECTION
# ============================================================


category_mean = (

    debits
    .groupby("category")["amount"]
    .transform("mean")
)


category_std = (

    debits
    .groupby("category")["amount"]
    .transform("std")
)


# ------------------------------------------------------------
# MANUAL Z-SCORE
# ------------------------------------------------------------

debits["z_score"] = np.where(

    category_std > 0,

    (
        debits["amount"]
        -
        category_mean
    )
    /
    category_std,

    0
)


# ------------------------------------------------------------
# FLAG ANOMALIES
# ------------------------------------------------------------

anomalies = (

    debits[
        debits["z_score"] > 2
    ]

    .sort_values(
        "z_score",
        ascending=False
    )

    .copy()
)


print("=" * 70)
print("FEATURE 7 - ANOMALY DETECTION")
print("=" * 70)

print(
    "Total anomalies:",
    len(anomalies)
)


print("\nTOP 5 ANOMALIES")

print("-" * 70)


for _, row in (
    anomalies
    .head(5)
    .iterrows()
):

    print(

        f"{row['date'].strftime('%d %b %Y')} | "
        f"{row['vendor_clean']:<22} | "
        f"{row['category']:<18} | "
        f"₹{row['amount']:>10,.2f} | "
        f"z={row['z_score']:.2f}"
    )

# Feature 8 – Spending Archetype Detection

SpendDNA uses quantitative rules to identify spending personalities.

The rules are implemented as separate functions so that each archetype can be tested independently.

Possible archetypes include:

- The Foodie
- The Quick Commerce Junkie
- The Shopaholic
- The Investor
- The Late-Night Snacker
- The Cab Commuter
- The Subscription Lover
- The YOLO Spender
- The Disciplined Saver

A custom Indian/Bengaluru-style archetype is also included as a bonus.

In [12]:
# ============================================================
# FEATURE 8 - SPENDING ARCHETYPE DETECTION
# ============================================================


# ------------------------------------------------------------
# THE FOODIE
# ------------------------------------------------------------

def foodie_rule(
    data,
    total_debit
):

    amount = data[
        data["category"].isin([

            "Food Delivery",

            "Restaurants",

            "Cafe"
        ])
    ]["amount"].sum()


    percentage = (

        amount /
        total_debit *
        100
    )


    return (
        percentage > 25,
        percentage
    )


# ------------------------------------------------------------
# QUICK COMMERCE JUNKIE
# ------------------------------------------------------------

def quick_commerce_rule(
    data,
    total_debit
):

    amount = data[
        data["category"] ==
        "Quick Commerce"
    ]["amount"].sum()


    percentage = (

        amount /
        total_debit *
        100
    )


    return (
        percentage > 15,
        percentage
    )


# ------------------------------------------------------------
# SHOPAHOLIC
# ------------------------------------------------------------

def shopaholic_rule(
    data,
    total_debit
):

    amount = data[
        data["category"] ==
        "E-commerce"
    ]["amount"].sum()


    percentage = (

        amount /
        total_debit *
        100
    )


    return (
        percentage > 15,
        percentage
    )


# ------------------------------------------------------------
# INVESTOR
# ------------------------------------------------------------

def investor_rule(
    data,
    total_debit
):

    amount = data[
        data["category"] ==
        "Investments"
    ]["amount"].sum()


    percentage = (

        amount /
        total_debit *
        100
    )


    return (
        percentage > 15,
        percentage
    )


# ------------------------------------------------------------
# LATE-NIGHT SNACKER
# ------------------------------------------------------------

def late_night_rule(data):

    food = data[
        data["category"] ==
        "Food Delivery"
    ]


    if len(food) == 0:

        return False, 0


    late = food[
        (
            food["hour"] >= 21
        )
        |
        (
            food["hour"] <= 2
        )
    ]


    percentage = (

        len(late) /
        len(food) *
        100
    )


    return (
        percentage > 50,
        percentage
    )


# ------------------------------------------------------------
# CAB COMMUTER
# ------------------------------------------------------------

def cab_commuter_rule(
    data,
    total_debit
):

    amount = data[
        data["category"] ==
        "Transport"
    ]["amount"].sum()


    percentage = (

        amount /
        total_debit *
        100
    )


    return (
        percentage > 10,
        percentage
    )


# ------------------------------------------------------------
# SUBSCRIPTION LOVER
# ------------------------------------------------------------

def subscription_rule(data):

    vendor_count = data[
        data["category"] ==
        "Subscriptions"
    ]["vendor_clean"].nunique()


    return (
        vendor_count >= 5,
        vendor_count
    )


# ------------------------------------------------------------
# YOLO SPENDER
# ------------------------------------------------------------

def yolo_rule(
    savings_rate
):

    return (
        savings_rate < 10,
        savings_rate
    )


# ------------------------------------------------------------
# DISCIPLINED SAVER
# ------------------------------------------------------------

def disciplined_saver_rule(
    savings_rate
):

    return (
        savings_rate > 40,
        savings_rate
    )


# ============================================================
# BONUS CUSTOM ARCHETYPE
# ============================================================
# THE CONVENIENCE SEEKER
#
# Rule:
# Food Delivery + Quick Commerce > 15%
# of total debit spending.
# ============================================================


def convenience_seeker_rule(
    data,
    total_debit
):

    amount = data[
        data["category"].isin([

            "Food Delivery",

            "Quick Commerce"
        ])
    ]["amount"].sum()


    percentage = (

        amount /
        total_debit *
        100
    )


    return (
        percentage > 15,
        percentage
    )


# ============================================================
# APPLY ALL RULES
# ============================================================


archetype_results = [

    (
        "THE FOODIE",

        foodie_rule(
            consumption,
            total_debits
        ),

        "% Food + Restaurant + Cafe"
    ),


    (
        "THE QUICK COMMERCE JUNKIE",

        quick_commerce_rule(
            consumption,
            total_debits
        ),

        "% Quick Commerce"
    ),


    (
        "THE SHOPAHOLIC",

        shopaholic_rule(
            consumption,
            total_debits
        ),

        "% E-commerce"
    ),


    (
        "THE INVESTOR",

        investor_rule(
            consumption,
            total_debits
        ),

        "% Investments"
    ),


    (
        "THE LATE-NIGHT SNACKER",

        late_night_rule(
            consumption
        ),

        "% Food Delivery late-night"
    ),


    (
        "THE CAB COMMUTER",

        cab_commuter_rule(
            consumption,
            total_debits
        ),

        "% Transport"
    ),


    (
        "THE SUBSCRIPTION LOVER",

        subscription_rule(
            consumption
        ),

        "subscription vendors"
    ),


    (
        "THE YOLO SPENDER",

        yolo_rule(
            savings_rate
        ),

        "savings rate"
    ),


    (
        "THE DISCIPLINED SAVER",

        disciplined_saver_rule(
            savings_rate
        ),

        "savings rate"
    ),


    (
        "THE CONVENIENCE SEEKER",

        convenience_seeker_rule(
            consumption,
            total_debits
        ),

        "% Food + Quick Commerce"
    )
]


print("=" * 70)
print("FEATURE 8 - SPENDING ARCHETYPES")
print("=" * 70)


detected_archetypes = []


for name, result, metric_name in archetype_results:

    applies = result[0]

    metric = result[1]


    if applies:

        detected_archetypes.append(
            name
        )


        if metric_name == "subscription vendors":

            metric_display = (
                f"{int(metric)} vendors"
            )

        else:

            metric_display = (
                f"{metric:.1f}%"
            )


        print(

            f"-> {name:<35}"
            f"{metric_display}"
        )


if len(detected_archetypes) == 0:

    print(
        "No archetype thresholds triggered."
    )

FEATURE 8 - SPENDING ARCHETYPES
-> THE SHOPAHOLIC                     35.4%
-> THE YOLO SPENDER                   -229.3%


# Bonus Features & Final SpendDNA Report

## Bonus Features

1. Day-of-week spending analysis
2. Vendor cleanup audit
3. Custom spending archetype
4. Three-month NumPy spending forecast

The final report combines the most important findings into a clean ASCII-style report suitable for screenshotting.

In [13]:
# ============================================================
# BONUS 1 - DAY OF WEEK ANALYSIS
# ============================================================


day_order = [

    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]


day_spending = (

    consumption
    .groupby("day_of_week")["amount"]
    .sum()
    .reindex(
        day_order,
        fill_value=0
    )
)


weekday_average = (
    day_spending[
        [
            "Monday",
            "Tuesday",
            "Wednesday",
            "Thursday",
            "Friday"
        ]
    ].mean()
)


weekend_average = (
    day_spending[
        [
            "Saturday",
            "Sunday"
        ]
    ].mean()
)


if weekday_average != 0:

    weekend_difference = (

        (
            weekend_average -
            weekday_average
        )
        /
        weekday_average
        *
        100
    )

else:

    weekend_difference = 0


print("=" * 70)
print("BONUS 1 - DAY-OF-WEEK ANALYSIS")
print("=" * 70)

print(
    day_spending.round(2)
)


print(
    f"\nAverage weekday spend: "
    f"₹{weekday_average:,.2f}"
)


print(
    f"Average weekend spend: "
    f"₹{weekend_average:,.2f}"
)


print(
    f"Weekend vs weekday: "
    f"{weekend_difference:+.1f}%"
)


# ============================================================
# BONUS 2 - VENDOR CLEANUP AUDIT
# ============================================================


print("\n" + "=" * 70)
print("BONUS 2 - VENDOR CLEANUP AUDIT")
print("=" * 70)


if len(uncategorised) == 0:

    print(
        "All descriptions were successfully mapped."
    )

else:

    print(
        f"Uncategorised descriptions: "
        f"{len(uncategorised)}"
    )

    for description in uncategorised:

        print(
            "-",
            description
        )


# ============================================================
# BONUS 3 - THREE MONTH FORECAST
# ============================================================


forecast_results = []


for category in monthly_matrix.index:

    values = (
        monthly_matrix
        .loc[category]
        .to_numpy(
            dtype=float
        )
    )


    last_three = values[-3:]


    forecast = np.mean(
        last_three
    )


    forecast_results.append({

        "category":
            category,

        "forecast":
            forecast
    })


forecast_df = pd.DataFrame(
    forecast_results
)


forecast_df = (
    forecast_df
    .sort_values(
        "forecast",
        ascending=False
    )
)


print("\n" + "=" * 70)
print("BONUS 3 - NEXT MONTH FORECAST")
print("=" * 70)


for _, row in forecast_df.iterrows():

    print(

        f"{row['category']:<20}"
        f"₹{row['forecast']:>12,.2f}"
    )

BONUS 1 - DAY-OF-WEEK ANALYSIS
day_of_week
Monday       242327.0
Tuesday      245368.0
Wednesday    292440.0
Thursday     183076.0
Friday       182080.0
Saturday     247028.0
Sunday       213716.0
Name: amount, dtype: float64

Average weekday spend: ₹229,058.20
Average weekend spend: ₹230,372.00
Weekend vs weekday: +0.6%

BONUS 2 - VENDOR CLEANUP AUDIT
Uncategorised descriptions: 1
- UPI-OLACABS@HDFCBANK

BONUS 3 - NEXT MONTH FORECAST
E-commerce          ₹  100,030.00
Investments         ₹   42,028.00
Utilities           ₹   24,796.67
Food Delivery       ₹   20,471.67
Restaurants         ₹   18,593.00
Quick Commerce      ₹   12,845.67
Groceries           ₹   11,929.00
Fuel                ₹   10,246.00
Transport           ₹    9,396.00
Cafe                ₹    6,011.33
Subscriptions       ₹    3,121.00
Entertainment       ₹    1,760.00


# Final Reflection

This project helped me understand how real-world transaction data differs from clean classroom datasets.

The main challenges were handling multiple date formats, different currency representations, duplicate transactions and inconsistent merchant descriptions.

I implemented a rule-based merchant normalisation system using dictionaries, loops and string methods without using regular expressions. I then categorised transactions, analysed monthly and time-of-day spending patterns, detected anomalous transactions using category-level z-scores and identified spending archetypes using quantitative rules.

The project demonstrated how Python fundamentals, NumPy and Pandas can be combined to create an end-to-end transaction analytics pipeline without machine learning or external fintech APIs.

## AI Assistance Disclosure

AI assistance was used for syntax support, debugging, code review and notebook structuring.

The dataset-specific vendor mappings and analytical results were verified using the provided transaction dataset.